In [79]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
# import glob
from PIL import Image
Image.MAX_IMAGE_PIXELS = 933120000
filename = "NDWI_Mask_130_resized.tif"
image_path = "./GEE_Masks/GEE_resized/"

import json

## Morphological Corruption

In [80]:
import os
import cv2 as cv
import numpy as np
import random
import shutil
import sys
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import rasterio
from glob import glob
import pandas as pd

In [81]:
def count_changed_pixels(original, eroded):
    return np.sum(original != eroded)

def apply_erosion_with_threshold(img, target_changes, kernel):
    eroded_img = img.copy()
    total_changed = 0
    iterations = 0
    total_changed_prev = 0
    
    while total_changed < target_changes:
        eroded_img_new = cv.erode(eroded_img, kernel, iterations=1)
        total_changed = count_changed_pixels(img, eroded_img_new)
        if (total_changed > target_changes):
            return eroded_img, total_changed_prev
        else:
            eroded_img = eroded_img_new
            total_changed_prev = total_changed
        
        iterations += 1
        if iterations > 100:  # Prevent infinite loop
            break
    
    return eroded_img, total_changed

# Function to apply dilation with stopping criteria based on the number of pixel changes
def apply_dilation_with_threshold(img, target_changes, kernel):
    dilated_img = img.copy()
    total_changed = 0
    iterations = 0
    total_changed_prev = 0

    while total_changed < target_changes:
        dilated_img_new = cv.dilate(dilated_img, kernel, iterations=1)
        total_changed = count_changed_pixels(img, dilated_img_new)
        if (total_changed > target_changes):
            return dilated_img_new, total_changed_prev
        else:
            dilated_img = dilated_img_new
            total_changed_prev = total_changed

        iterations += 1
        if iterations > 100:  # Prevent infinite loop
            break

    return dilated_img, total_changed

In [82]:
def min_corrupted_pixels(set_1, set_2, kernel, combined_erosion, combined_dilation):
    # Initialize variables to accumulate the sum of corrupted pixels
    min_changed_pixels_erosion = sys.maxsize
    min_changed_pixels_dilation = sys.maxsize
    
    # Lists to store corrupted pixels for plotting
    erosion_pixel_counts = []
    dilation_pixel_counts = []

    # Loop through all images in set_1
    for img_name in set_1:
        img_path = os.path.join(combined_erosion, img_name)
        img = cv.imread(img_path, cv.IMREAD_GRAYSCALE)
        assert img is not None, f"File {img_name} could not be read"

        # Apply erosion and count changed pixels
        eroded_img = cv.erode(img, kernel, iterations=1)
        num_changed_pixels = count_changed_pixels(img, eroded_img)
        erosion_pixel_counts.append(num_changed_pixels)
        min_changed_pixels_erosion = min(num_changed_pixels, min_changed_pixels_erosion)

    # Loop through all images in set_2
    for img_name in set_2:
        img_path = os.path.join(combined_dilation, img_name)
        img = cv.imread(img_path, cv.IMREAD_GRAYSCALE)
        assert img is not None, f"File {img_name} could not be read"

        # Apply dilation and count changed pixels
        dilated_img = cv.dilate(img, kernel, iterations=1)
        num_changed_pixels = count_changed_pixels(img, dilated_img)
        dilation_pixel_counts.append(num_changed_pixels)
        min_changed_pixels_dilation = min(num_changed_pixels, min_changed_pixels_dilation)

    # # Plot the number of corrupted pixels for each image
    # plt.figure(figsize=(10, 5))
    # plt.plot(erosion_pixel_counts, label='Erosion')
    # plt.plot(dilation_pixel_counts, label='Dilation')
    # plt.xlabel('Image Index')
    # plt.ylabel('Number of Corrupted Pixels')
    # plt.title('Number of Corrupted Pixels per Image')
    # plt.legend()
    # plt.show()

    return min_changed_pixels_erosion, min_changed_pixels_dilation

In [83]:
def max_corrupted_pixels(set_1, set_2, kernel, combined_erosion, combined_dilation):
    # Initialize variables to accumulate the sum of corrupted pixels
    max_changed_pixels_erosion = 0
    max_changed_pixels_dilation = 0

    # Loop through all images in set_1
    for img_name in set_1:
        img_path = os.path.join(combined_erosion, img_name)
        img = cv.imread(img_path, cv.IMREAD_GRAYSCALE)
        assert img is not None, f"File {img_name} could not be read"

        # Apply erosion and count changed pixels
        eroded_img = cv.erode(img, kernel, iterations=1)
        num_changed_pixels = count_changed_pixels(img, eroded_img)
        # print(num_changed_pixels)
        max_changed_pixels_erosion = max(num_changed_pixels,max_changed_pixels_erosion)

    # Loop through all images in set_2
    for img_name in set_2:
        img_path = os.path.join(combined_dilation, img_name)
        img = cv.imread(img_path, cv.IMREAD_GRAYSCALE)
        assert img is not None, f"File {img_name} could not be read"

        # Apply dilation and count changed pixels
        dilated_img = cv.dilate(img, kernel, iterations=1)
        num_changed_pixels = count_changed_pixels(img, dilated_img)
        # print(num_changed_pixels)
        max_changed_pixels_dilation = max(num_changed_pixels,max_changed_pixels_dilation)

    # Calculate the average number of corrupted pixels
    # avg_changed_pixels_erosion = max_changed_pixels_erosion / len(set_1)
    # avg_changed_pixels_dilation = max_changed_pixels_dilation / len(set_2)

    return max_changed_pixels_erosion, max_changed_pixels_dilation

In [84]:
def choose_kernel_dilation(white_pixel_ratio):
    if white_pixel_ratio > 0.4:
        kernel_size = np.random.choice([3, 5, 7], p=[0.7, 0.15, 0.15])
    elif 0.2 < white_pixel_ratio <= 0.4:
        kernel_size = np.random.choice([3, 5, 7], p=[0.15, 0.7, 0.15])
    else:
        kernel_size = np.random.choice([3, 5, 7], p=[0.15, 0.15, 0.7])
    
    return np.ones((kernel_size, kernel_size), np.uint8)

def choose_kernel_erosion(white_pixel_ratio):
    if white_pixel_ratio > 0.4:
        kernel_size = np.random.choice([3, 5, 7], p=[0.15, 0.15, 0.7])
    elif 0.2 < white_pixel_ratio <= 0.4:
        kernel_size = np.random.choice([3, 5, 7], p=[0.15, 0.7, 0.15])
    else:
        kernel_size = np.random.choice([3, 5, 7], p=[0.7, 0.15, 0.15])
    
    return np.ones((kernel_size, kernel_size), np.uint8)

In [85]:
corruption = 15
image_dim = 512

source_dir = '.\\GEE_Masks\\GEE_resized\\train_gee\\train_gee'
target_dir = f'.\\GEE_Masks\\GEE_resized\\train_gee\\train_{corruption}_gee_mixed'
print(target_dir)
# Step 1: Get all .tif files in the directory
tif_files = glob(os.path.join(source_dir, '*_resized_corrupt.tif'))

os.makedirs(target_dir,exist_ok=True)
# Step 2: Split files into 87.5% training and 12.5% validation
# print(tif_files)
# train_files, val_files = train_test_split(tif_files, test_size=0.125, random_state=42)
train_files = tif_files
# Step 3: Further split train_files into three equal sets
third = len(train_files) // 3
set_1 = train_files[:third]       # For erosion
set_2 = train_files[third:2*third]  # For dilation
set_3 = train_files[2*third:]     # For normal (copy without modification)
# print(set_1)
# Step 4: Apply erosion to the first group

# Kernel for erosion
# kernel = np.ones((5, 5), np.uint8)

df = pd.read_csv("results.csv")

threshold_erosion = int(corruption*image_dim*image_dim)/100  # Example threshold; adjust as needed
threshold_dilation = int(corruption*image_dim*image_dim)/100

print(threshold_dilation)
print(threshold_erosion)

print(len(set_1))
print(len(set_2))
print(len(set_3))

.\GEE_Masks\GEE_resized\train_gee\train_15_gee_mixed
39321.6
39321.6
336
336
338


In [86]:
os.makedirs(target_dir, exist_ok=True)

for file_name in set_1:
    img = cv.imread(file_name, cv.IMREAD_GRAYSCALE)
    assert img is not None, f"File {file_name} could not be read"

    # Apply erosion
    file_name_new = os.path.basename(file_name).replace("_corrupt", "")
    # print(df.loc[df['Image'] == file_name_new, 'White Pixel Ratio'])
    white_pixel_ratio = df.loc[df['Image'] == file_name_new, 'White Pixel Ratio'].values[0]
    kernel = choose_kernel_erosion(white_pixel_ratio)
    erosion, changed_pixels = apply_erosion_with_threshold(img, threshold_erosion, kernel)

    # Save the eroded image with the new filename
    # new_filename = os.path.basename(file_name).replace('_resized.tif', '_resized_corrupt.tif')
    cv.imwrite(os.path.join(target_dir, os.path.basename(file_name)), erosion)

    print(f'{file_name} (Erosion): {changed_pixels} pixels changed')

# Step 5: Apply dilation to the second group
for file_name in set_2:
    img = cv.imread(file_name, cv.IMREAD_GRAYSCALE)
    assert img is not None, f"File {file_name} could not be read"

    # Apply dilation
    file_name_new = os.path.basename(file_name).replace("_corrupt", "")
    # print(df.loc[df['Image'] == file_name_new, 'White Pixel Ratio'])
    white_pixel_ratio = df.loc[df['Image'] == file_name_new, 'White Pixel Ratio'].values[0]
    kernel = choose_kernel_dilation(white_pixel_ratio)
    dilation, changed_pixels = apply_dilation_with_threshold(img, threshold_dilation, kernel)

    # Save the dilated image with the new filename
    # new_filename = os.path.basename(file_name).replace('_resized.tif', '_resized_corrupt.tif')
    cv.imwrite(os.path.join(target_dir, os.path.basename(file_name)), dilation)

    print(f'{file_name} (Dilation): {changed_pixels} pixels changed')

# Step 6: Copy the third group without modification
for file_name in set_3:
    src_path = file_name
    dest_path = os.path.join(target_dir, os.path.basename(file_name))
    shutil.copy(src_path, dest_path)
    print(f'{file_name} copied to {dest_path}')


# for file_name in val_files:
#     src_path = file_name
#     dest_path = os.path(target_dir,os.path.basename(file_name).replace('resized.tif','_resized_corrupt.tif'))
#     shutil.copy(src_path,dest_path)

# print('Processing completed.')

.\GEE_Masks\GEE_resized\train_gee\train_gee\NDWI_Mask_0_resized_corrupt.tif (Erosion): 26545 pixels changed
.\GEE_Masks\GEE_resized\train_gee\train_gee\NDWI_Mask_1000_resized_corrupt.tif (Erosion): 920 pixels changed
.\GEE_Masks\GEE_resized\train_gee\train_gee\NDWI_Mask_1001_resized_corrupt.tif (Erosion): 24231 pixels changed
.\GEE_Masks\GEE_resized\train_gee\train_gee\NDWI_Mask_1002_resized_corrupt.tif (Erosion): 10986 pixels changed
.\GEE_Masks\GEE_resized\train_gee\train_gee\NDWI_Mask_1003_resized_corrupt.tif (Erosion): 38474 pixels changed
.\GEE_Masks\GEE_resized\train_gee\train_gee\NDWI_Mask_1004_resized_corrupt.tif (Erosion): 35799 pixels changed
.\GEE_Masks\GEE_resized\train_gee\train_gee\NDWI_Mask_1005_resized_corrupt.tif (Erosion): 38772 pixels changed
.\GEE_Masks\GEE_resized\train_gee\train_gee\NDWI_Mask_1006_resized_corrupt.tif (Erosion): 21198 pixels changed
.\GEE_Masks\GEE_resized\train_gee\train_gee\NDWI_Mask_1007_resized_corrupt.tif (Erosion): 4886 pixels changed
.\GEE_M

In [87]:
set_1 = train_files
target_dir = f'.\\GEE_Masks\\GEE_resized\\train_gee\\train_{corruption}_gee_erosion'
os.makedirs(target_dir, exist_ok=True)

for file_name in set_1:
    img = cv.imread(file_name, cv.IMREAD_GRAYSCALE)
    assert img is not None, f"File {file_name} could not be read"

    # Apply erosion
    file_name_new = os.path.basename(file_name).replace("_corrupt", "")
    # print(df.loc[df['Image'] == file_name_new, 'White Pixel Ratio'])
    white_pixel_ratio = df.loc[df['Image'] == file_name_new, 'White Pixel Ratio'].values[0]
    kernel = choose_kernel_erosion(white_pixel_ratio)
    erosion, changed_pixels = apply_erosion_with_threshold(img, threshold_erosion, kernel)

    # Save the eroded image with the new filename
    # new_filename = os.path.basename(file_name).replace('_resized.tif', '_resized_corrupt.tif')
    cv.imwrite(os.path.join(target_dir, os.path.basename(file_name)), erosion)

    print(f'{file_name} (Erosion): {changed_pixels} pixels changed')

set_2 = train_files
target_dir = f'.\\GEE_Masks\\GEE_resized\\train_gee\\train_{corruption}_gee_dilation'
os.makedirs(target_dir, exist_ok=True)

# Step 5: Apply dilation to the second group
for file_name in set_2:
    img = cv.imread(file_name, cv.IMREAD_GRAYSCALE)
    assert img is not None, f"File {file_name} could not be read"

    # Apply dilation
    file_name_new = os.path.basename(file_name).replace("_corrupt", "")
    # print(df.loc[df['Image'] == file_name_new, 'White Pixel Ratio'])
    white_pixel_ratio = df.loc[df['Image'] == file_name_new, 'White Pixel Ratio'].values[0]
    kernel = choose_kernel_dilation(white_pixel_ratio)
    dilation, changed_pixels = apply_dilation_with_threshold(img, threshold_dilation, kernel)

    # Save the dilated image with the new filename
    # new_filename = os.path.basename(file_name).replace('_resized.tif', '_resized_corrupt.tif')
    cv.imwrite(os.path.join(target_dir, os.path.basename(file_name)), dilation)

    print(f'{file_name} (Dilation): {changed_pixels} pixels changed')

.\GEE_Masks\GEE_resized\train_gee\train_gee\NDWI_Mask_0_resized_corrupt.tif (Erosion): 26545 pixels changed
.\GEE_Masks\GEE_resized\train_gee\train_gee\NDWI_Mask_1000_resized_corrupt.tif (Erosion): 920 pixels changed
.\GEE_Masks\GEE_resized\train_gee\train_gee\NDWI_Mask_1001_resized_corrupt.tif (Erosion): 24231 pixels changed
.\GEE_Masks\GEE_resized\train_gee\train_gee\NDWI_Mask_1002_resized_corrupt.tif (Erosion): 10986 pixels changed
.\GEE_Masks\GEE_resized\train_gee\train_gee\NDWI_Mask_1003_resized_corrupt.tif (Erosion): 34407 pixels changed
.\GEE_Masks\GEE_resized\train_gee\train_gee\NDWI_Mask_1004_resized_corrupt.tif (Erosion): 35799 pixels changed
.\GEE_Masks\GEE_resized\train_gee\train_gee\NDWI_Mask_1005_resized_corrupt.tif (Erosion): 38772 pixels changed
.\GEE_Masks\GEE_resized\train_gee\train_gee\NDWI_Mask_1006_resized_corrupt.tif (Erosion): 21198 pixels changed
.\GEE_Masks\GEE_resized\train_gee\train_gee\NDWI_Mask_1007_resized_corrupt.tif (Erosion): 4886 pixels changed
.\GEE_M